# Day 12: UART RX, SPI & IP Integration

## Pre-Class Videos (~50 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | UART RX — Oversampling | ~15 min | `d12_s1_uart_rx_oversampling.html` |
| 2 | UART RX Implementation | ~15 min | `d12_s2_uart_rx_implementation.html` |
| 3 | SPI Protocol | ~12 min | `d12_s3_spi_protocol.html` |
| 4 | IP Integration Philosophy | ~8 min | `d12_s4_ip_integration.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day12_ex01_uart_rx.v` | UART RX with 16× oversampling, built-in 2-FF sync, self-checking TB |
| `code/day12_ex02_uart_loopback.v` | RX → TX echo top module for Go Board integration test |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d12_oversampling.svg` | 16× oversampling: 16 ticks per bit, center sampling at tick 7-8 |

## Key Concepts
- RX must discover timing (TX controls it)
- 16× oversampling: center-of-bit sampling for noise immunity
- Start-bit verification: wait to center, re-check for glitches
- Loopback test = gold standard integration verification
- SPI: 4 wires, synchronous, full duplex, 4 modes (CPOL/CPHA)
- IP integration: read spec → wrapper → synchronizers → testbench → verify

## Week 3 Summary

Your module library after Week 3:
`rom_sync`, `ram_sp`, `pattern_sequencer`, `top_pll_demo`, `uart_tx`, `uart_rx`, `uart_loopback`
+ all Week 1-2 modules.

Complete communication stack: FPGA ↔ PC via UART. Ready for Week 4.

## Directory Structure

```
day12_uart_rx_spi_ip_integration/
├── d12_s1_uart_rx_oversampling.html
├── d12_s2_uart_rx_implementation.html
├── d12_s3_spi_protocol.html
├── d12_s4_ip_integration.html
├── code/
│   ├── day12_ex01_uart_rx.v
│   └── day12_ex02_uart_loopback.v
├── diagrams/
│   └── d12_oversampling.svg
├── day12_quiz.md
└── day12_readme.md
```

---
## Code Examples

### `day12_ex01_uart_rx.v`

```verilog
// =============================================================================
// day12_ex01_uart_rx.v — UART Receiver (8N1, 16× Oversampling)
// Day 12: UART RX, SPI & IP Integration
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Built-in 2-FF synchronizer on i_rx. 16× oversampling for center-of-bit
// sampling. Outputs o_valid pulse for one cycle when a byte is received.
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day12_ex01_uart_rx.v && vvp sim
// Synth:  yosys -p "read_verilog day12_ex01_uart_rx.v; synth_ice40 -top uart_rx"
// =============================================================================

module uart_rx #(
    `ifdef SIMULATION
    parameter CLK_FREQ  = 1_600,       // tiny for fast sim
    parameter BAUD_RATE = 100
    `else
    parameter CLK_FREQ  = 25_000_000,
    parameter BAUD_RATE = 115_200
    `endif
)(
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_rx,           // serial input (external, async)
    output reg  [7:0] o_data,         // received byte
    output reg        o_valid         // one-cycle pulse when byte ready
);

    // ---- Oversampling Parameters ----
    localparam CLKS_PER_BIT  = CLK_FREQ / BAUD_RATE;
    localparam OS_RATE       = 16;
    localparam CLKS_PER_TICK = CLKS_PER_BIT / OS_RATE;
    localparam TICK_W        = $clog2(CLKS_PER_TICK + 1);

    // ---- State Encoding ----
    localparam S_IDLE  = 2'd0;
    localparam S_START = 2'd1;
    localparam S_DATA  = 2'd2;
    localparam S_STOP  = 2'd3;

    reg [1:0]       r_state, r_next_state;

    // ---- Built-in 2-FF Synchronizer ----
    reg r_rx_sync0, r_rx_sync1;
    always @(posedge i_clk) begin
        r_rx_sync0 <= i_rx;
        r_rx_sync1 <= r_rx_sync0;
    end

    // ---- Oversample Tick Generator ----
    reg [TICK_W-1:0] r_tick_cnt;
    wire w_os_tick = (r_tick_cnt == CLKS_PER_TICK - 1);

    always @(posedge i_clk) begin
        if (i_reset || r_state == S_IDLE)
            r_tick_cnt <= 0;
        else if (w_os_tick)
            r_tick_cnt <= 0;
        else
            r_tick_cnt <= r_tick_cnt + 1;
    end

    // ---- Oversample Counter (0..15 per bit) ----
    reg [3:0] r_os_cnt;

    // ---- Bit Index (0..7) ----
    reg [2:0] r_bit_idx;

    // ---- Data Shift Register ----
    reg [7:0] r_data;

    // ============================================================
    // Block 1 — State Register
    // ============================================================
    always @(posedge i_clk) begin
        if (i_reset)
            r_state <= S_IDLE;
        else
            r_state <= r_next_state;
    end

    // ============================================================
    // Block 2 — Next-State Logic
    // ============================================================
    always @(*) begin
        r_next_state = r_state;

        case (r_state)
            S_IDLE: begin
                if (!r_rx_sync1)            // falling edge: possible start
                    r_next_state = S_START;
            end
            S_START: begin
                if (w_os_tick && r_os_cnt == 7) begin
                    if (!r_rx_sync1)         // verified at center
                        r_next_state = S_DATA;
                    else
                        r_next_state = S_IDLE; // glitch
                end
            end
            S_DATA: begin
                if (w_os_tick && r_os_cnt == 15 && r_bit_idx == 7)
                    r_next_state = S_STOP;
            end
            S_STOP: begin
                if (w_os_tick && r_os_cnt == 15)
                    r_next_state = S_IDLE;
            end
            default: r_next_state = S_IDLE;
        endcase
    end

    // ---- Oversample Counter & Bit Index Management ----
    always @(posedge i_clk) begin
        if (i_reset) begin
            r_os_cnt  <= 0;
            r_bit_idx <= 0;
            r_data    <= 0;
            o_data    <= 0;
            o_valid   <= 0;
        end else begin
            o_valid <= 0;   // default: clear valid pulse

            case (r_state)
                S_IDLE: begin
                    r_os_cnt  <= 0;
                    r_bit_idx <= 0;
                end

                S_START: begin
                    if (w_os_tick)
                        r_os_cnt <= r_os_cnt + 1;
                    // Reset on entering DATA
                    if (w_os_tick && r_os_cnt == 7 && !r_rx_sync1) begin
                        r_os_cnt  <= 0;
                        r_bit_idx <= 0;
                    end
                end

                S_DATA: begin
                    if (w_os_tick) begin
                        if (r_os_cnt == 7) begin
                            // Center of bit — sample!
                            r_data[r_bit_idx] <= r_rx_sync1;
                        end
                        if (r_os_cnt == 15) begin
                            r_os_cnt <= 0;
                            r_bit_idx <= r_bit_idx + 1;
                        end else
                            r_os_cnt <= r_os_cnt + 1;
                    end
                end

                S_STOP: begin
                    if (w_os_tick) begin
                        if (r_os_cnt == 7) begin
                            // Center of stop bit — output the byte
                            o_data  <= r_data;
                            o_valid <= 1;
                        end
                        if (r_os_cnt == 15)
                            r_os_cnt <= 0;
                        else
                            r_os_cnt <= r_os_cnt + 1;
                    end
                end
            endcase
        end
    end

endmodule

// =============================================================================
// Self-Checking Testbench — Uses a TX model to drive the RX
// =============================================================================
`ifdef SIMULATION
module tb_uart_rx;

    localparam CLK_FREQ  = 1_600;
    localparam BAUD_RATE = 100;
    localparam CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;  // 16

    reg  clk = 0, reset = 1;
    wire [7:0] rx_data;
    wire       rx_valid;

    // ---- Serial line (driven by our bit-banger) ----
    reg r_serial = 1;  // idle high

    uart_rx #(
        .CLK_FREQ(CLK_FREQ),
        .BAUD_RATE(BAUD_RATE)
    ) uut (
        .i_clk(clk), .i_reset(reset),
        .i_rx(r_serial),
        .o_data(rx_data), .o_valid(rx_valid)
    );

    always #5 clk = ~clk;

    integer test_count = 0, fail_count = 0;

    // ---- Bit-Banger: Transmit a byte on r_serial ----
    task send_byte;
        input [7:0] b;
        integer i;
    begin
        // Start bit
        r_serial = 0;
        repeat (CLKS_PER_BIT) @(posedge clk);
        // Data bits (LSB first)
        for (i = 0; i < 8; i = i + 1) begin
            r_serial = b[i];
            repeat (CLKS_PER_BIT) @(posedge clk);
        end
        // Stop bit
        r_serial = 1;
        repeat (CLKS_PER_BIT) @(posedge clk);
        // Inter-frame gap
        repeat (CLKS_PER_BIT) @(posedge clk);
    end
    endtask

    task check_rx;
        input [7:0] expected;
        input [8*20-1:0] name;
    begin
        test_count = test_count + 1;
        // Wait for valid pulse (with timeout)
        repeat (CLKS_PER_BIT * 12) @(posedge clk);
        if (rx_data !== expected) begin
            $display("FAIL: %0s — expected %h, got %h", name, expected, rx_data);
            fail_count = fail_count + 1;
        end else
            $display("PASS: %0s — received %h ('%c')", name, rx_data, rx_data);
    end
    endtask

    initial begin
        $dumpfile("tb_uart_rx.vcd");
        $dumpvars(0, tb_uart_rx);

        $display("\n=== UART RX Testbench ===\n");

        repeat(10) @(posedge clk);
        reset = 0;
        repeat(10) @(posedge clk);

        // Test 1: 'H' (0x48)
        fork
            send_byte(8'h48);
            check_rx(8'h48, "Byte 'H'");
        join

        // Test 2: 0xA5
        fork
            send_byte(8'hA5);
            check_rx(8'hA5, "Byte 0xA5");
        join

        // Test 3: 0x00
        fork
            send_byte(8'h00);
            check_rx(8'h00, "Byte 0x00");
        join

        // Test 4: 0xFF
        fork
            send_byte(8'hFF);
            check_rx(8'hFF, "Byte 0xFF");
        join

        $display("\n=== SUMMARY ===");
        $display("Tests: %0d  Passed: %0d  Failed: %0d",
                 test_count, test_count - fail_count, fail_count);
        if (fail_count == 0)
            $display("\n*** ALL TESTS PASSED ***\n");
        else
            $display("\n*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

### `day12_ex02_uart_loopback.v`

```verilog
// =============================================================================
// day12_ex02_uart_loopback.v — UART RX → TX Echo (Loopback)
// Day 12: UART RX, SPI & IP Integration
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Gold-standard integration test: type on PC terminal → see echo.
// Also displays the received byte on 7-seg and LEDs.
// =============================================================================
// Synth:  yosys -p "read_verilog day12_ex02_uart_loopback.v uart_tx.v uart_rx.v \
//          hex_to_7seg.v; synth_ice40 -top uart_loopback"
// =============================================================================

module uart_loopback #(
    parameter CLK_FREQ  = 25_000_000,
    parameter BAUD_RATE = 115_200
)(
    input  wire       i_clk,
    input  wire       i_rx,          // UART RX from FTDI
    output wire       o_tx,          // UART TX to FTDI
    output wire [3:0] o_led,         // lower nibble of last byte
    output wire [6:0] o_seg1,        // upper nibble hex display
    output wire [6:0] o_seg2         // lower nibble hex display
);

    wire [7:0] w_rx_data;
    wire       w_rx_valid;
    wire       w_tx_busy;

    // ---- UART Receiver ----
    uart_rx #(
        .CLK_FREQ(CLK_FREQ),
        .BAUD_RATE(BAUD_RATE)
    ) rx_inst (
        .i_clk   (i_clk),
        .i_reset (1'b0),
        .i_rx    (i_rx),
        .o_data  (w_rx_data),
        .o_valid (w_rx_valid)
    );

    // ---- UART Transmitter (echo) ----
    // Hold received byte until TX is free
    reg [7:0] r_tx_data;
    reg       r_tx_pending;

    always @(posedge i_clk) begin
        if (w_rx_valid) begin
            r_tx_data    <= w_rx_data;
            r_tx_pending <= 1;
        end else if (r_tx_pending && !w_tx_busy) begin
            r_tx_pending <= 0;
        end
    end

    wire w_tx_valid = r_tx_pending && !w_tx_busy;

    uart_tx #(
        .CLK_FREQ(CLK_FREQ),
        .BAUD_RATE(BAUD_RATE)
    ) tx_inst (
        .i_clk   (i_clk),
        .i_reset (1'b0),
        .i_valid (w_tx_valid),
        .i_data  (r_tx_data),
        .o_busy  (w_tx_busy),
        .o_tx    (o_tx)
    );

    // ---- Display last received byte ----
    reg [7:0] r_display;
    always @(posedge i_clk)
        if (w_rx_valid) r_display <= w_rx_data;

    assign o_led = ~r_display[3:0];   // active-low LEDs

    // 7-segment decoders (if available in your library)
    // hex_to_7seg seg_hi (.i_hex(r_display[7:4]), .o_seg(o_seg1));
    // hex_to_7seg seg_lo (.i_hex(r_display[3:0]), .o_seg(o_seg2));

    // Placeholder if hex_to_7seg is not yet wired:
    assign o_seg1 = 7'h7F;
    assign o_seg2 = 7'h7F;

endmodule
```

---
## Pre-Class Self-Check Quiz

**Q1:** Why does the UART RX use 16× oversampling instead of 1× sampling?

<details><summary>Answer</summary>
Without a shared clock, the RX doesn't know exactly when each bit starts. 16× oversampling provides 16 sample points per bit, allowing center-of-bit sampling (tick 7-8) for maximum noise immunity and baud rate mismatch tolerance.
</details>

**Q2:** What is the loopback test and why is it the gold standard?

<details><summary>Answer</summary>
RX → TX echo: type characters on PC terminal, see them echoed back. If the echo works, both TX and RX are verified correct with a single test. It validates the complete communication path end-to-end.
</details>

**Q3:** How many wires does SPI use? What advantage does it have over UART?

<details><summary>Answer</summary>
4 wires: SCLK, MOSI, MISO, CS_N. The master provides the clock, so there's no baud rate negotiation and no oversampling needed. SPI is synchronous and can run at much higher speeds (10+ Mbps vs. ~1 Mbps for UART).
</details>

**Q4:** What's the first step on the IP integration checklist?

<details><summary>Answer</summary>
**Read the interface specification.** Understand the ports, protocols, timing requirements, and configuration options before writing any wrapper or testbench code.
</details>